In [ ]:
import pandas as pd

# Define the standard column names for C-MAPSS
columns = ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
          ['sensor_measurement_{}'.format(i) for i in range(1, 22)]

def process_and_save_train_data(input_file, output_file, column_names):
    # 1. Load the raw text file
    df = pd.read_csv(input_file, sep='\s+', header=None, names=column_names)

    # 2. Calculate the maximum cycle for each unit_number
    # This creates a series where the index is the unit_number and value is the max cycle
    max_cycle = df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_cycle.columns = ['unit_number', 'max_cycle']

    # 3. Merge the max_cycle back into the original dataframe
    df = df.merge(max_cycle, on='unit_number', how='left')

    # 4. Calculate RUL: (Max Cycle - Current Cycle)
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']

    # 5. Drop the helper 'max_cycle' column if you don't need it
    df.drop('max_cycle', axis=1, inplace=True)

    # Save to CSV
    df.to_csv(output_file, index=False)
    print(f"Successfully processed {input_file} with RUL and saved to {output_file}")

# Execute the processing for the training set
process_and_save_train_data('/content/train_FD001.txt', 'train_FD001.csv', columns)

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3887461557.py:9: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(input_file, sep='\s+', header=None, names=column_names)


Successfully processed /content/train_FD001.txt with RUL and saved to train_FD001.csv


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# 1. Load Data
# Ensure these files are in your current working directory
train_df = pd.read_csv('/content/train_FD001.csv')
test_df = pd.read_csv('/content/test_FD001.csv')
# Corrected: Load RUL_FD001.txt which contains the true RUL values for the test set
rul_true = pd.read_csv('/content/RUL_FD001.txt', header=None, names=['remaining_useful_life'])

# 2. Data Preprocessing

# Drop constant columns that do not carry information
drop_cols = ['op_setting_3', 'sensor_measurement_1', 'sensor_measurement_5',
             'sensor_measurement_10', 'sensor_measurement_16', 'sensor_measurement_18',
             'sensor_measurement_19']

train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

# Define feature columns (excluding identifiers and target)
features = [c for c in train_df.columns if c not in ['unit_number', 'time_in_cycles', 'RUL']]

# Clip RUL at 125 (piece-wise linear degradation assumption)
# This helps the model focus on the degradation phase
train_df['RUL'] = train_df['RUL'].clip(upper=125)

# Normalize features to range [0, 1]
scaler = MinMaxScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

# 3. Sequence Generation (Sliding Window)
window_size = 30


def gen_train_sequences(df, window_size, feature_cols):
    """
    Generates sliding window sequences for training.
    """
    X = []
    y = []
    for unit in df['unit_number'].unique():
        unit_data = df[df['unit_number'] == unit]
        data_arr = unit_data[feature_cols].values
        rul_arr = unit_data['RUL'].values

        # Create sliding windows
        for i in range(len(data_arr) - window_size + 1):
            X.append(data_arr[i : i + window_size])
            y.append(rul_arr[i + window_size - 1])

    return np.array(X), np.array(y)

def gen_test_sequences(df, window_size, feature_cols):
    """
    Generates the LAST sequence for each unit in the test set.
    """
    X = []
    for unit in df['unit_number'].unique():
        unit_data = df[df['unit_number'] == unit]
        # We take the last 'window_size' data points to predict the RUL at the end
        if len(unit_data) >= window_size:
            data_arr = unit_data[feature_cols].values
            X.append(data_arr[-window_size:])
    return np.array(X)

# Generate data
X_train, y_train = gen_train_sequences(train_df, window_size, features)
X_test = gen_test_sequences(test_df, window_size, features)
y_test_true = rul_true['remaining_useful_life'].values

print(f"Training Data Shape: {X_train.shape}")
print(f"Test Data Shape: {X_test.shape}")

# 4. Build 1D CNN Model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(window_size, len(features))),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(100, activation='relu'),
    Dropout(0.2), # Dropout to reduce overfitting
    Dense(1) # Output layer for regression
])

# Compile model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])

# 5. Train Model
# Validating on 20% of training data to monitor performance
history = model.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.2, verbose=1)

# 6. Evaluate on Test Set
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test_true, y_pred)

print("-" * 30)
print(f"Final Mean Absolute Error on Test Set: {mae:.4f}")
print("-" * 30)

Training Data Shape: (17731, 30, 17)
Test Data Shape: (100, 30, 17)
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


222/222 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 3092.3872 - mae: 43.8857 - val_loss: 571.0315 - val_mae: 19.4170
Epoch 2/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 507.3289 - mae: 17.7561 - val_loss: 444.1355 - val_mae: 16.4567
Epoch 3/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 424.8197 - mae: 15.8624 - val_loss: 332.5168 - val_mae: 14.1633
Epoch 4/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 322.3844 - mae: 13.8429 - val_loss: 252.9953 - val_mae: 12.7008
Epoch 5/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 286.1181 - mae: 13.1418 - val_loss: 236.4509 - val_mae: 12.1108
Epoch 6/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 272.5483 - mae: 12.8082 - val_loss: 233.6102 - val_mae: 11.9582
Epoch 7/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 262.5094 - mae: 12.4178 - val_loss: 235.8308 - val_mae: 11.6796
Epoch 8/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 263.3420 - mae: 12.4973 - val_loss: 228.7961 - val_mae: 11.7507
Epoch 9/50
2

In [14]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# 1. Load Data
train_df = pd.read_csv('train_FD001.csv')
test_df = pd.read_csv('test_FD001.csv')
rul_true = pd.read_csv('RUL_FD001.csv') # Has header 'remaining_useful_life'

# 2. Prepare Test Data RUL
# Calculate RUL for test data to make it usable for training
# RUL at time t = (True RUL at end) + (Max Cycle - Current Cycle)

# Get max cycle for each unit in test set
max_cycles_test = test_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycles_test.columns = ['unit_number', 'max_cycle']

# Merge with true RUL values
# Assuming rul_true rows correspond to unit_numbers 1, 2, 3...
rul_true['unit_number'] = rul_true.index + 1
test_rul_info = pd.merge(max_cycles_test, rul_true, on='unit_number')

# Merge back into test_df
test_df = pd.merge(test_df, test_rul_info, on='unit_number')

# Calculate RUL column
test_df['RUL'] = test_df['remaining_useful_life'] + test_df['max_cycle'] - test_df['time_in_cycles']

# Drop helper columns to match train_df structure
test_df.drop(columns=['max_cycle', 'remaining_useful_life'], inplace=True)

# 3. Combine Datasets
# CRITICAL: Re-index unit_number in test_df so they don't overlap with train_df
# Train has units 1-100. We make Test units 101-200.
test_df['unit_number'] = test_df['unit_number'] + train_df['unit_number'].max()

# Concatenate
full_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

print(f"Combined Dataset Shape: {full_df.shape}")
print(f"Total Units: {full_df['unit_number'].nunique()}")

# 4. Preprocessing (Same as before)
drop_cols = ['op_setting_3', 'sensor_measurement_1', 'sensor_measurement_5',
             'sensor_measurement_10', 'sensor_measurement_16', 'sensor_measurement_18',
             'sensor_measurement_19']
full_df.drop(columns=drop_cols, inplace=True)

features = [c for c in full_df.columns if c not in ['unit_number', 'time_in_cycles', 'RUL']]

# Clip RUL
full_df['RUL'] = full_df['RUL'].clip(upper=125)

# Normalize
scaler = MinMaxScaler()
full_df[features] = scaler.fit_transform(full_df[features])

# 5. Sequence Generation
window_size = 30

def gen_sequences(df, window_size, feature_cols):
    X = []
    y = []
    for unit in df['unit_number'].unique():
        unit_data = df[df['unit_number'] == unit]
        data_arr = unit_data[feature_cols].values
        rul_arr = unit_data['RUL'].values

        if len(data_arr) < window_size:
            continue

        for i in range(len(data_arr) - window_size + 1):
            X.append(data_arr[i : i + window_size])
            y.append(rul_arr[i + window_size - 1])

    return np.array(X), np.array(y)

# Generate sequences from the FULL dataset
X, y = gen_sequences(full_df, window_size, features)
print(f"Total Sequences: {X.shape}")

# 6. Build & Train Model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(window_size, len(features))),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(100, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])

# Train on everything (no validation split since we are using all data for final model)
# Or keep a small split just to see if it's learning
model.fit(X, y, epochs=50, batch_size=64, validation_split=0.1, verbose=1)

# Save
model.save('rul_model_full.h5')

Combined Dataset Shape: (33727, 27)
Total Units: 200
Total Sequences: (27927, 30, 17)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - loss: 2658.1221 - mae: 39.3727 - val_loss: 477.9848 - val_mae: 17.3112
Epoch 2/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 479.7124 - mae: 17.1762 - val_loss: 454.2007 - val_mae: 16.9270
Epoch 3/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 408.9588 - mae: 15.6368 - val_loss: 304.3053 - val_mae: 13.8230
Epoch 4/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 329.2382 - mae: 14.0548 - val_loss: 275.9883 - val_mae: 12.3730
Epoch 5/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 298.9126 - mae: 13.4171 - val_loss: 389.3869 - val_mae: 16.0512
Epoch 6/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 289.5765 - mae: 13.1876 - val_loss: 299.2166 - val_mae: 13.2252
Epoch 7/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 287.2554 - mae: 13.1730 - val_loss: 335.2321 - val_mae: 14.5264
Epoch 8/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 279.8489 - mae: 12.9369 - val_loss: 330.1695 - val_mae: 14.2780
Ep

In [15]:
import joblib

# 1. Save the Keras model
model.save('rul_model.h5')

# 2. Save the fitted Scaler (CRITICAL step)
# The web app must use this exact scaler to normalize new data
joblib.dump(scaler, 'scaler.pkl')

print("Artifacts saved: rul_model.h5 and scaler.pkl")

Artifacts saved: rul_model.h5 and scaler.pkl
